# I-Turn AI server (Google Colab)

This notebook runs **only the AI model**, behind a small HTTP API. The I-Turn
app itself (UI, backend, database) runs on your laptop and talks to this
notebook over the internet through a tunnel. The app needs no GPU, no CUDA and
no model weights.

```
  your laptop                                       this Colab runtime
  ┌────────────────────┐   https://xxxx.trycloudflare.com   ┌──────────────────┐
  │ I-Turn UI + backend│ ─────────────────────────────────► │ AI server + model│
  └────────────────────┘        AI_BASE_URL / AI_API_KEY     └──────────────────┘
```

**Steps**

1. **Runtime → Change runtime type → GPU** (a free T4 is enough for a 3B model).
2. Optionally change the model in the *Configuration* cell.
3. **Runtime → Run all.** Loading the model takes a few minutes the first time.
4. Copy the two lines printed at the bottom (`AI_BASE_URL=…` and `AI_API_KEY=…`)
   into the `.env` file of your I-Turn checkout, then start I-Turn as usual.

> **Privacy.** While this is running, whatever the app sends to the model
> travels over a public tunnel to a Google-hosted machine. Use it with test
> data, not with real students' conversations, unless your ethics approval
> covers it. The tunnel URL is unguessable and every request needs the API key,
> but stop the runtime when you're done.


## 1 · Configuration

This is the only cell you should need to edit.

**Choosing a model.** Set `MODEL_NAME` to any Hugging Face instruction-tuned
chat model that fits the GPU. The server normalises whatever the model produces
into the same API response, so I-Turn doesn't care which one you pick.

| Model | Notes |
|---|---|
| `Qwen/Qwen2.5-3B-Instruct` | default; fits a free T4 comfortably |
| `Qwen/Qwen2.5-7B-Instruct` | set `LOAD_IN_4BIT = True` on a T4 |
| `mistralai/Mistral-7B-Instruct-v0.3` | set `LOAD_IN_4BIT = True` on a T4 |
| `meta-llama/Llama-3.2-3B-Instruct` | gated: accept the licence on Hugging Face, then set `HF_TOKEN` |
| `google/gemma-2-2b-it` | gated: as above |

`HF_TOKEN` can also be stored as a Colab **secret** named `HF_TOKEN` (key icon in
the left sidebar) so it never appears in the notebook.


In [ ]:
#@title Configuration
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"  #@param {type:"string"}
PORT = 8000                              #@param {type:"integer"}
LOAD_IN_4BIT = False                     #@param {type:"boolean"}
API_KEY = ""                             #@param {type:"string"}
HF_TOKEN = ""                            #@param {type:"string"}
# API_KEY: leave empty and a random one is generated and printed for you.
# HF_TOKEN: only for gated models (Llama, Gemma); or use a Colab secret.


## 2 · Implementation

Nothing below needs editing. It installs dependencies, starts the AI server,
loads the model, opens a tunnel and prints the values to put in your `.env`.


In [ ]:
# torch and CUDA are preinstalled in Colab; only the serving/model libraries are needed.
!pip install -q -U fastapi "uvicorn[standard]" httpx pydantic "transformers>=4.44" accelerate bitsandbytes


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU is attached to this runtime. Use Runtime -> Change runtime type -> GPU, "
        "then Runtime -> Run all again."
    )
print("GPU:", torch.cuda.get_device_name(0),
      f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")


In [ ]:
%%writefile ai_server.py
"""
I-Turn AI server — the only place in the project where a model is loaded.

The application (app/) never imports torch or transformers. It sends HTTP
requests to this server, or to anything else that implements the same contract.
This file is the reference implementation of that contract. It is deliberately a
single file so the Colab notebook can embed it verbatim; collab/build_notebook.py
keeps the two in sync.

CONTRACT (version 1)
--------------------
POST /v1/generate
    {
      "prompt":         "..."                     # a single user message, OR
      "messages":       [{"role": "user|assistant|system", "content": "..."}],
      "system_prompt":  "..."                     # optional
      "temperature":    0.7,   "top_p": 0.9,   "max_tokens": 1024,
      "repeat_penalty": 1.0,   "json_mode": false,
      "model":          "..."                     # advisory; see below
    }
    ->
    {
      "text": "...", "model": "...", "finish_reason": "stop" | "length",
      "usage": {"input_tokens": 0, "output_tokens": 0}
    }

GET /health
    {"status": "ok" | "loading" | "error", "model": "...", "engine": "...", ...}
    200 when ready, 503 otherwise.

Errors are always {"error": {"code": "...", "message": "..."}} with codes:
    unauthorized, model_loading, model_error, out_of_memory, generation_failed

The model that answers is whichever one this server loaded. A `model` field in
the request is advisory: it is logged if it differs, and the response always
says which model actually produced the text. Whatever the underlying model
emits, the response has exactly this shape — the application never needs to know
which model it is talking to.

ENGINES
-------
The engine is how this server runs a model. It is chosen by AI_ENGINE:
    transformers  Hugging Face model on this machine's GPU (Colab). Default.
    ollama        Forwards to a local Ollama instance.
    stub          No model. Canned text. For UI/tests on a laptop with no GPU.

CONFIGURATION (environment)
---------------------------
    AI_ENGINE     transformers | ollama | stub          (default: transformers)
    MODEL_NAME    HF repo id, or Ollama tag              (default: Qwen/Qwen2.5-3B-Instruct)
    AI_API_KEY    if set, every request must send "Authorization: Bearer <key>"
    HOST / PORT   where to listen                        (default: 127.0.0.1:8001)
    LOAD_IN_4BIT  "1" to load the transformers model in 4-bit (needs bitsandbytes)
    OLLAMA_URL    (default: http://127.0.0.1:11434)

Run:  python -m ai_server.server          (from the repository root)
"""

from __future__ import annotations

import hmac
import logging
import os
import threading
from contextlib import asynccontextmanager
from dataclasses import dataclass
from typing import Any, Literal

import httpx
from fastapi import Depends, FastAPI, Header, Request
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field, model_validator

log = logging.getLogger("ai_server")

CONTRACT_VERSION = "1"
DEFAULT_MODEL = "Qwen/Qwen2.5-3B-Instruct"


# ---------------------------------------------------------------------------
# Wire format
# ---------------------------------------------------------------------------

class Message(BaseModel):
    role: Literal["system", "user", "assistant"]
    content: str = Field(min_length=1)


class GenerateRequest(BaseModel):
    prompt: str | None = Field(default=None, min_length=1)
    messages: list[Message] | None = Field(default=None, min_length=1, max_length=200)
    system_prompt: str | None = None
    model: str | None = None
    temperature: float = Field(0.7, ge=0.0, le=2.0)
    top_p: float = Field(0.9, gt=0.0, le=1.0)
    max_tokens: int = Field(1024, ge=1, le=4096)
    repeat_penalty: float = Field(1.0, ge=1.0, le=2.0)   # 1.0 = off
    json_mode: bool = False

    @model_validator(mode="after")
    def _one_input(self) -> "GenerateRequest":
        if (self.prompt is None) == (self.messages is None):
            raise ValueError("send exactly one of 'prompt' or 'messages'")
        return self

    def chat_messages(self) -> list[dict[str, str]]:
        out: list[dict[str, str]] = []
        if self.system_prompt:
            out.append({"role": "system", "content": self.system_prompt})
        if self.messages is not None:
            out.extend(m.model_dump() for m in self.messages)
        else:
            out.append({"role": "user", "content": self.prompt or ""})
        return out


class Usage(BaseModel):
    input_tokens: int = 0
    output_tokens: int = 0


class GenerateResponse(BaseModel):
    text: str
    model: str
    finish_reason: Literal["stop", "length"] = "stop"
    usage: Usage = Usage()


# ---------------------------------------------------------------------------
# Engines
# ---------------------------------------------------------------------------

@dataclass
class Params:
    temperature: float
    top_p: float
    max_tokens: int
    repeat_penalty: float
    json_mode: bool


@dataclass
class Completion:
    text: str
    finish_reason: str = "stop"
    input_tokens: int = 0
    output_tokens: int = 0


class EngineError(Exception):
    """A failure inside an engine, already classified with a contract code."""

    def __init__(self, code: str, message: str):
        self.code, self.message = code, message
        super().__init__(f"{code}: {message}")


class Engine:
    name = "base"

    def __init__(self, model: str):
        self.model = model

    def load(self) -> None:
        """Blocking; may take minutes. Raise EngineError('model_error', ...) on failure."""

    def generate(self, messages: list[dict[str, str]], p: Params) -> Completion:
        raise NotImplementedError


class StubEngine(Engine):
    """No model at all. Lets the whole application run on a laptop with nothing
    installed but the app's own dependencies."""

    name = "stub"

    def generate(self, messages: list[dict[str, str]], p: Params) -> Completion:
        text = ("{}" if p.json_mode else
                "[stub] I hear you. Tell me a bit more about how that's been going.")
        return Completion(text=text, output_tokens=len(text.split()),
                          input_tokens=sum(len(m["content"].split()) for m in messages))


class OllamaEngine(Engine):
    """Forwards to a local Ollama. Keeps a local-GPU workflow that already uses
    Ollama working behind the same contract."""

    name = "ollama"

    def __init__(self, model: str, url: str = "http://127.0.0.1:11434"):
        super().__init__(model)
        self.url = url.rstrip("/")
        # One pooled client: a fresh connection per call costs ~0.7s on Windows
        # (~2.7s via "localhost"), which dwarfs generation time on a small model.
        self._http = httpx.Client(timeout=300, limits=httpx.Limits(keepalive_expiry=60.0))

    def load(self) -> None:
        try:
            tags = self._http.get(f"{self.url}/api/tags", timeout=10).json()
        except (httpx.HTTPError, ValueError) as e:
            raise EngineError("model_error", f"Ollama not reachable at {self.url}: {e}") from e
        names = {m.get("name") for m in tags.get("models", [])}
        if self.model not in names:
            raise EngineError("model_error",
                              f"Ollama has no model {self.model!r}. Run: ollama pull {self.model}")

    def generate(self, messages: list[dict[str, str]], p: Params) -> Completion:
        payload: dict[str, Any] = {
            "model": self.model,
            "messages": messages,
            "stream": False,
            "options": {"temperature": p.temperature, "top_p": p.top_p,
                        "num_predict": p.max_tokens, "repeat_penalty": p.repeat_penalty},
        }
        if p.json_mode:
            payload["format"] = "json"
        try:
            r = self._http.post(f"{self.url}/api/chat", json=payload)
            r.raise_for_status()
            body = r.json()
        except (httpx.HTTPError, ValueError) as e:
            raise EngineError("generation_failed", f"Ollama request failed: {e}") from e
        return Completion(
            text=body["message"]["content"],
            finish_reason="length" if body.get("done_reason") == "length" else "stop",
            input_tokens=int(body.get("prompt_eval_count", 0)),
            output_tokens=int(body.get("eval_count", 0)),
        )


class TransformersEngine(Engine):
    """Any Hugging Face causal LM that ships a chat template."""

    name = "transformers"

    def __init__(self, model: str, load_in_4bit: bool = False):
        super().__init__(model)
        self.load_in_4bit = load_in_4bit
        self._tok = self._model = self._torch = None
        self._eos: set[int] = set()

    def load(self) -> None:
        try:
            import torch
            from transformers import AutoModelForCausalLM, AutoTokenizer

            if not torch.cuda.is_available():
                log.warning("No GPU visible to torch. Generation will be very slow. "
                            "In Colab: Runtime -> Change runtime type -> GPU.")
            # T4 (Colab's free GPU) has no fast bfloat16; newer GPUs prefer it.
            dtype = (torch.bfloat16
                     if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
                     else torch.float16 if torch.cuda.is_available() else torch.float32)
            kwargs: dict[str, Any] = {"torch_dtype": dtype, "device_map": "auto"}
            if self.load_in_4bit:
                from transformers import BitsAndBytesConfig
                kwargs["quantization_config"] = BitsAndBytesConfig(
                    load_in_4bit=True, bnb_4bit_compute_dtype=dtype)

            log.info("Loading %s (dtype=%s, 4bit=%s)", self.model, dtype, self.load_in_4bit)
            self._tok = AutoTokenizer.from_pretrained(self.model)
            self._model = AutoModelForCausalLM.from_pretrained(self.model, **kwargs)
            self._model.eval()
            self._torch = torch

            eos = self._model.generation_config.eos_token_id
            self._eos = set(eos if isinstance(eos, list) else [eos]) if eos is not None else set()
            if self._tok.eos_token_id is not None:
                self._eos.add(self._tok.eos_token_id)
        except Exception as e:  # noqa: BLE001 — any failure here means "no model"
            oom = "out of memory" in str(e).lower()
            hint = " (GPU out of memory: pick a smaller model or set LOAD_IN_4BIT)" if oom else ""
            raise EngineError("model_error", f"{type(e).__name__}: {e}{hint}") from e

    def _render(self, messages: list[dict[str, str]]) -> str:
        tok = self._tok
        try:
            return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:  # noqa: BLE001
            # Some templates (Gemma, some Mistral) reject a system role. Fold it
            # into the first user turn instead of failing.
            system = "\n\n".join(m["content"] for m in messages if m["role"] == "system")
            rest = [dict(m) for m in messages if m["role"] != "system"]
            if system and rest and rest[0]["role"] == "user":
                rest[0]["content"] = f"{system}\n\n{rest[0]['content']}"
            return tok.apply_chat_template(rest, tokenize=False, add_generation_prompt=True)

    def generate(self, messages: list[dict[str, str]], p: Params) -> Completion:
        torch, tok, model = self._torch, self._tok, self._model
        text = self._render(messages)
        # The template already contains any BOS token; don't add a second one.
        inputs = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)
        gen: dict[str, Any] = {
            "max_new_tokens": p.max_tokens,
            "repetition_penalty": p.repeat_penalty,
            "pad_token_id": tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id,
        }
        if p.temperature > 0:
            gen.update(do_sample=True, temperature=p.temperature, top_p=p.top_p)
        else:
            gen.update(do_sample=False)
        try:
            with torch.no_grad():
                out = model.generate(**inputs, **gen)
        except torch.cuda.OutOfMemoryError as e:
            torch.cuda.empty_cache()
            raise EngineError("out_of_memory", "GPU ran out of memory during generation") from e

        n_in = inputs["input_ids"].shape[1]
        new = out[0][n_in:]
        hit_eos = len(new) > 0 and int(new[-1]) in self._eos
        return Completion(
            text=tok.decode(new, skip_special_tokens=True),
            finish_reason="stop" if hit_eos or len(new) < p.max_tokens else "length",
            input_tokens=int(n_in),
            output_tokens=int(len(new)),
        )


def build_engine(name: str, model: str) -> Engine:
    if name == "stub":
        return StubEngine(model or "stub")
    if name == "ollama":
        return OllamaEngine(model, os.environ.get("OLLAMA_URL", "http://127.0.0.1:11434"))
    if name == "transformers":
        return TransformersEngine(model, load_in_4bit=os.environ.get("LOAD_IN_4BIT") == "1")
    raise ValueError(f"unknown AI_ENGINE {name!r} (use transformers, ollama or stub)")


# ---------------------------------------------------------------------------
# App
# ---------------------------------------------------------------------------

class ServerError(Exception):
    def __init__(self, status: int, code: str, message: str):
        self.status, self.code, self.message = status, code, message


def create_app(engine: Engine, api_key: str = "") -> FastAPI:
    state: dict[str, str] = {"status": "loading", "detail": ""}
    gpu_lock = threading.Lock()      # one generation at a time on one GPU

    def load_in_background() -> None:
        try:
            engine.load()
            state["status"] = "ok"
            log.info("Model ready: %s (%s)", engine.model, engine.name)
        except EngineError as e:
            state.update(status="error", detail=e.message)
            log.error("Model failed to load: %s", e.message)
        except Exception as e:  # noqa: BLE001
            state.update(status="error", detail=f"{type(e).__name__}: {e}")
            log.exception("Model failed to load")

    @asynccontextmanager
    async def lifespan(_: FastAPI):
        # Serve /health immediately and load in the background, so a caller can
        # tell "still loading" from "nothing there".
        threading.Thread(target=load_in_background, daemon=True).start()
        yield

    app = FastAPI(title="I-Turn AI server", lifespan=lifespan)

    def require_key(authorization: str | None = Header(default=None)) -> None:
        if not api_key:
            return
        if not authorization or not hmac.compare_digest(authorization, f"Bearer {api_key}"):
            raise ServerError(401, "unauthorized", "missing or invalid API key")

    @app.exception_handler(ServerError)
    async def _server_error(_: Request, exc: ServerError) -> JSONResponse:
        return JSONResponse(status_code=exc.status,
                            content={"error": {"code": exc.code, "message": exc.message}})

    @app.get("/health")
    def health(_: None = Depends(require_key)) -> JSONResponse:
        body: dict[str, Any] = {"status": state["status"], "model": engine.model,
                                "engine": engine.name, "contract": CONTRACT_VERSION}
        if state["detail"]:
            body["detail"] = state["detail"]
        return JSONResponse(status_code=200 if state["status"] == "ok" else 503, content=body)

    @app.post("/v1/generate", response_model=GenerateResponse)
    def generate(req: GenerateRequest, _: None = Depends(require_key)) -> GenerateResponse:
        if state["status"] == "loading":
            raise ServerError(503, "model_loading", "the model is still loading")
        if state["status"] == "error":
            raise ServerError(503, "model_error", state["detail"] or "the model failed to load")
        if req.model and req.model != engine.model:
            log.warning("Request asked for model %r; serving %r", req.model, engine.model)

        params = Params(req.temperature, req.top_p, req.max_tokens,
                        req.repeat_penalty, req.json_mode)
        try:
            with gpu_lock:
                c = engine.generate(req.chat_messages(), params)
        except EngineError as e:
            status = 503 if e.code == "out_of_memory" else 500
            raise ServerError(status, e.code, e.message) from e
        except Exception as e:  # noqa: BLE001
            log.exception("Generation failed")
            raise ServerError(500, "generation_failed", f"{type(e).__name__}") from e

        return GenerateResponse(
            text=(c.text or "").strip(),
            model=engine.model,
            finish_reason="length" if c.finish_reason == "length" else "stop",
            usage=Usage(input_tokens=c.input_tokens, output_tokens=c.output_tokens),
        )

    return app


def main() -> None:
    import uvicorn

    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
    engine = build_engine(os.environ.get("AI_ENGINE", "transformers"),
                          os.environ.get("MODEL_NAME", DEFAULT_MODEL))
    app = create_app(engine, api_key=os.environ.get("AI_API_KEY", ""))
    uvicorn.run(app, host=os.environ.get("HOST", "127.0.0.1"),
                port=int(os.environ.get("PORT", "8001")), log_level="info")


if __name__ == "__main__":
    main()


In [ ]:
import errno, json, os, secrets, socket, subprocess, sys, time, urllib.error, urllib.request

LOAD_TIMEOUT_MIN = 30


def _stop(proc):
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(10)
        except subprocess.TimeoutExpired:
            proc.kill()


def _port_in_use(port):
    # Free only if the connection is actively refused; anything else (accepted,
    # or a busy listener that stops answering) means something is already there.
    with socket.socket() as sock:
        sock.settimeout(4)                    # Windows can take ~2s to report a refusal
        return sock.connect_ex(("127.0.0.1", port)) not in (errno.ECONNREFUSED, 10061)


_stop(globals().get("server_proc"))          # so this cell can be re-run safely
for _ in range(8):                            # give the old one a moment to release the port
    if not _port_in_use(PORT):
        break
    time.sleep(0.5)
else:
    raise RuntimeError(
        f"Port {PORT} is already in use, probably by an earlier run of this notebook. "
        "Change PORT in the Configuration cell, or use Runtime -> Restart session and run again.")

AI_API_KEY = API_KEY.strip() or secrets.token_urlsafe(24)
env = {
    **os.environ,
    "AI_ENGINE": "transformers",
    "MODEL_NAME": MODEL_NAME,
    "HOST": "127.0.0.1",
    "PORT": str(PORT),
    "AI_API_KEY": AI_API_KEY,
    "LOAD_IN_4BIT": "1" if LOAD_IN_4BIT else "0",
}
hf_token = HF_TOKEN.strip()
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN") or ""
    except Exception:
        pass
if hf_token:
    env["HF_TOKEN"] = hf_token

server_proc = subprocess.Popen(
    [sys.executable, "ai_server.py"], env=env,
    stdout=open("ai_server.log", "w"), stderr=subprocess.STDOUT,
)


def local_health():
    req = urllib.request.Request(
        f"http://127.0.0.1:{PORT}/health", headers={"Authorization": f"Bearer {AI_API_KEY}"})
    try:
        with urllib.request.urlopen(req, timeout=5) as r:
            return json.load(r)
    except urllib.error.HTTPError as e:      # 503 while loading / failed still carries a status body
        try:
            return json.load(e)
        except Exception:
            return None
    except Exception:
        return None


def _log_tail():
    return open("ai_server.log").read()[-3000:]


print(f"Starting the AI server and loading {MODEL_NAME} (first run downloads the weights)...")
deadline, last = time.time() + LOAD_TIMEOUT_MIN * 60, None
while time.time() < deadline:
    if server_proc.poll() is not None:
        print(_log_tail())
        raise RuntimeError("The AI server process exited. The log above says why.")
    h = local_health()
    status = (h or {}).get("status")
    if isinstance(h, dict) and isinstance(h.get("error"), dict) and h["error"].get("code") == "unauthorized":
        raise RuntimeError("Something else is answering on this port with a different API key. "
                           "Change PORT, or Runtime -> Restart session, and run again.")
    if status != last:
        print(f"[{time.strftime('%H:%M:%S')}] server status: {status or 'starting'}")
        last = status
    if status == "ok":
        break
    if status == "error":
        print(_log_tail())
        raise RuntimeError(f"The model failed to load: {h.get('detail')}")
    time.sleep(3)
else:
    print(_log_tail())
    raise TimeoutError(f"The model did not finish loading within {LOAD_TIMEOUT_MIN} minutes.")

print("Model loaded:", h["model"])


In [ ]:
import re

# A Cloudflare "quick tunnel": free, no account, gives a public https URL.
CLOUDFLARED = "./cloudflared"
CLOUDFLARED_URL = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"

if not os.path.exists(CLOUDFLARED):
    for attempt in range(1, 4):
        try:
            urllib.request.urlretrieve(CLOUDFLARED_URL, CLOUDFLARED)
            os.chmod(CLOUDFLARED, 0o755)
            break
        except Exception as e:
            print(f"cloudflared download failed (attempt {attempt}/3): {e}")
            time.sleep(3)
    else:
        raise RuntimeError("Could not download cloudflared from GitHub. Check Colab's internet "
                           "access and re-run this cell.")
print(subprocess.run([CLOUDFLARED, "--version"], capture_output=True, text=True).stdout.strip())

PUBLIC_URL = None
for attempt in range(1, 4):
    _stop(globals().get("tunnel_proc"))
    tunnel_proc = subprocess.Popen(
        [CLOUDFLARED, "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
        stdout=open("cloudflared.log", "w"), stderr=subprocess.STDOUT,
    )
    for _ in range(45):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("cloudflared.log").read())
        if m:
            PUBLIC_URL = m.group(0)
            break
        if tunnel_proc.poll() is not None:
            break
        time.sleep(1)
    if PUBLIC_URL:
        break
    print(f"No tunnel URL yet (attempt {attempt}/3); trying again...")
    time.sleep(3)

if not PUBLIC_URL:
    print(open("cloudflared.log").read()[-3000:])
    raise RuntimeError("cloudflared did not produce a public URL after 3 tries. The log above "
                       "says why (a 429 means Cloudflare is rate-limiting free tunnels; wait a "
                       "few minutes and re-run this cell).")

# The URL can take a few seconds to start resolving; confirm it reaches the server.
reachable = False
for _ in range(30):
    try:
        req = urllib.request.Request(
            PUBLIC_URL + "/health",
            headers={"Authorization": f"Bearer {AI_API_KEY}", "User-Agent": "i-turn-notebook"})
        with urllib.request.urlopen(req, timeout=10) as r:
            reachable = r.status == 200
        if reachable:
            break
    except Exception:
        pass
    time.sleep(2)
print("Tunnel is reachable from the internet." if reachable else
      "Tunnel URL created but not answering yet; give it a minute before using it.")


In [ ]:
def show_banner():
    print("=" * 60)
    print("i-turn AI SERVER")
    print("=" * 60)
    print()
    print("Local endpoint:")
    print(f"http://127.0.0.1:{PORT}")
    print()
    print("Public endpoint:")
    print(PUBLIC_URL)
    print()
    print("Model:", MODEL_NAME)
    print()
    print("Set this in your i-turn .env:")
    print()
    print(f"AI_BASE_URL={PUBLIC_URL}")
    print(f"AI_API_KEY={AI_API_KEY}")
    print("=" * 60)


show_banner()


## 3 · Keep it running (optional)

The server keeps running after the cells above finish, for as long as this
runtime stays connected. This cell just watches it and reprints the values;
interrupt it (■) when you're done. Colab may disconnect an idle runtime, and the
public URL changes every time the tunnel restarts, so update `AI_BASE_URL` if
you re-run.


In [ ]:
try:
    while True:
        h = local_health()
        tunnel_up = tunnel_proc.poll() is None
        print(f"[{time.strftime('%H:%M:%S')}] server: {(h or {}).get('status', 'DOWN')}"
              f" | tunnel: {'up' if tunnel_up else 'DOWN'}")
        time.sleep(60)
except KeyboardInterrupt:
    show_banner()
